### Bill Scraper: Intial Local Scrape

This code does a local scrape of all of the available bills, before I set up the updater.

See the notebook: DD_Day5_PA_Bill_Scraper_Updates_Only.ipynb

For a further explanation

In [ ]:
###FUNCTION FOR EXTRACTING DATA POINTS FROM EACH BILL PAGE 
##THIS IS USING BEAUTIFULSOUP METHODS TO TARGET SPECIFIC TAGS AND/OR TEXT HEADERS

import re
from bs4 import BeautifulSoup

def extract_data_points(bill_page,url,hash_id,bill_id):
    fields = {}
    fields["content_hash"]=hash_id
    fields["url"]=url
    fields["bill_id"]=link.text
    fields["bill_name"]=bill_page.find(class_='h1 header-title').text.strip()
    fields["short_des"]=bill_page.find(id="shortTitle-wrapper").text.strip()
    if bill_page.find(class_="col-12 ps-2"):
        fields["memo_name"]=bill_page.find(class_="col-12 ps-2").text.strip()
        fields["memo_link"]=bill_page.find(class_="col-12 ps-2").a['href']
    prime_container = bill_page.find(string="Prime Sponsor").parent.find_next_sibling('div')
    fields["prime_sponsor"]={
        "name": prime_container.strong.text.strip(),
        "party": prime_container.span.text.strip(),
        "district": prime_container.span.next_sibling.strip(),
    }
    if bill_page.find(string="Co-Sponsors"):
        co_sponsors=bill_page.find(string="Co-Sponsors").parent.find_next_sibling('div')
        all_cosp=co_sponsors.find_all(class_=re.compile("^col"))
        fields["cosponsors"]=[]
        for cosp in all_cosp:
            each_cosp= {
            "name": cosp.strong.text.strip(),
            "party": cosp.span.text.strip(),
            "district": cosp.span.next_sibling.strip()
            }
            fields["cosponsors"].append(each_cosp)
    last_action=bill_page.find(string=re.compile("Last Action")).parent.parent.text.strip()
    fields["last_action"]=" ".join(last_action.split())
    timeline=bill_page.find(class_=re.compile("timeline"))
    all_items = timeline.find_all('span')
    time_dict = {}
    for item in all_items:
        item_html = BeautifulSoup(item['title'], "html.parser")
        time_elements = item_html.find_all(string=True)
        time_key = time_elements.pop(0)
        time_dict[time_key] = time_elements
    fields["timeline"] =time_dict
    history_table=bill_page.find(string=re.compile("View Full History")).parent.parent.find_next_sibling('div').table
    actions_list = []
    for row in history_table.find_all('tr'):
        actions_dict = {}
        actions_dict['action']=row.text.strip()
        if row.td.a:
            actions_dict['doc']=row.td.a['aria-label']
            actions_dict['pdf_link']=row.td.a['href']
            actions_dict['text_link']="https://www.palegis.us" + row.td.a['href'].replace("text/PDF/","text/HTM/")
            # actions_dict['text_link']=row.td.a['href']
        actions_list.append(actions_dict)
    fields["actions"]=actions_list
    main_votes=bill_page.find(id="VotesSection").find_next_sibling('div').find_all(class_="card-body")
    fields["votes"] = []
    for vote in main_votes:
        vt={}
        vt['date']=vote.find(class_=re.compile("^h5")).text
        vt['type']=vote.find(class_=re.compile("^h6")).text
        vt['bill']=vote.find(class_=re.compile("^mb-3")).text
        vt['link']=vote.a['href']
        vote_tally=vote.find_all(class_=re.compile("^fa-solid"))
        for item in vote_tally:
            vt[item.find_next_siblings('span')[1].text]=item.find_next_siblings('span')[0].text
        fields["votes"].append(vt)
    if bill_page.find(id="billVotes"):
        add_votes=bill_page.find(id="billVotes").table.find_all('tr')
        for vote in add_votes:
            vt={}
            date_type=vote.find(class_=re.compile("^h6")).text
            vt['date']=date_type.split("-")[0].strip()
            vt['type']=date_type.split("-")[1].strip()
            vt['bill']=vote.a.text
            vt['link']=vote.a['href']
            vt['YES Votes']=vote.find(class_=re.compile("fa-square-check")).parent.text
            vt['NO Votes']=vote.find(class_=re.compile("fa-square-xmark")).parent.text
            fields["votes"].append(vt)
    return fields

In [ ]:
####SCRAPING THE TEXT OF THE BILL ITSELF####

def get_bill_text(text_url,head):
    print(text_url)
    bill_response = requests.get(
        url=text_url,
        headers=head,
        timeout=120
    )
    bill_html = bill_response.content
    bill_doc = BeautifulSoup(bill_html, "html.parser")
    return bill_doc.find(id="page-container").text

In [ ]:

########################SCRAPING LOGIC##############################

#########IMPORTS#########

import os
import json
import hashlib
import datetime
import traceback
import requests
import time
import re
from bs4 import BeautifulSoup


#########STEP ONE#########
#check for data history


DATA_FILE = "data/pa_senate_bills.json" #FINAL OUTPUT OF SCRAPED DATA
TODAY_STR = datetime.date.today().isoformat()

# CHECKING / BUILDING DIRECTORIES FOR DATA, CHANGE LOG, AND ERROR
os.makedirs("data/changelogs", exist_ok=True)
os.makedirs("data/error_logs", exist_ok=True)
os.makedirs(os.path.dirname(DATA_FILE), exist_ok=True)

if os.path.exists(DATA_FILE):
    print("Found existing dataset. Loading history...")
    with open(DATA_FILE, 'r') as f:
        yesterdays_list = json.load(f)
    # Map by URL for instant lookup
    old_data_map = {item['url']: item for item in yesterdays_list}
else:
    print("No existing dataset found. Initializing a baseline run...")
    old_data_map = {} # Empty map forces EVERYTHING to be treated as a new entry


#########STEP TWO#########
#Set up Change Log and Error Log
    
changelog = {
    "date": TODAY_STR,
    "additions": [],
    "deletions": [],
    "modifications": []
}
error_log = {
    "date": TODAY_STR,
    "errors": []
}


#########STEP THREE#########
#scrape contents page get the current contents

todays_bills = []
#creating a header for the request--this mimicks the info that a browser sends

HEAD = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.9',
    'Accept-Encoding': 'gzip, deflate, br',
    'Connection': 'keep-alive',
    'Upgrade-Insecure-Requests': '1',
    'Sec-Fetch-Dest': 'document',
    'Sec-Fetch-Mode': 'navigate',
    'Sec-Fetch-Site': 'none',
    'Sec-Fetch-User': '?1',
}
url = "https://www.palegis.us/legislation/bills/bill-index?display=index&sessYr=2025&sessInd=0&billBody=S&filter=bills"
raw_html = requests.get(url,headers=HEAD).content
contents_doc = BeautifulSoup(raw_html, "html.parser")
bill_container = contents_doc.h3.parent.parent
all_bill_links = bill_container.find_all('a')
all_bill_urls = ["https://www.palegis.us"+item['href'] for item in all_bill_links]





#########STEP FOUR#########
#checking/scraping the bill pages

for link in all_bill_links:#EXAMPLE CURRENTLY JUST SCRAPE THE FIRST 5 PAGES
    url = "https://www.palegis.us"+link['href']
    yesterdays_item = old_data_map.get(url)
    time.sleep(1)
    try:
        raw_html = requests.get(url,headers=HEAD).content
        bill_page = BeautifulSoup(raw_html, "html.parser")
        ###setting up hash for change detection
        history_table=bill_page.find(string=re.compile("View Full History")).parent.parent.find_next_sibling('div').table
        last_action=bill_page.find(string=re.compile("Last Action")).parent.parent.text.strip()
        hash_string =" ".join(last_action.split())+" ".join(history_table.text.split())[-50:]
        print(hash_string.lower())
        hash_id=hashlib.md5(hash_string.lower().encode('utf-8')).hexdigest()
        #IF THIS IS A NEW LINK--JUST GET EVERYTHING
        if url not in old_data_map:
            ###SEND THE CONTENT AND KEY DATA POINTS TO THE FUNCTION
            print("new entry")
            bill_dict=extract_data_points(bill_page,url,hash_id,link.text)
            bill_dict["bill_text"]=get_bill_text(bill_dict["actions"][0]["text_link"],HEAD)
            # print(bill_dict["actions"][0]["text_link"])
            todays_bills.append(bill_dict)
            changelog["additions"].append({"bill_id": link.text, "url": url, "action": bill_dict["last_action"]})
        else:
            yesterdays_item = old_data_map[url]
            yesterdays_hash = yesterdays_item['content_hash']
            #CHECK HASHES FOR CHANGE
            if yesterdays_hash == hash_id: #NO CHANGE
                print("they match!")
                todays_bills.append(yesterdays_item)
            else:
                print("no match") #THERE ARE CHANGES
                bill_dict=extract_data_points(bill_page,url,hash_id,link.text)
                ##get text of bill##
                bill_dict["bill_text"]=get_bill_text(bill_dict["actions"][0]["text_link"],head)
                todays_bills.append(bill_dict)
                meaningful_changes = {}
                for key, value in bill_dict.items():
                    if yesterdays_item.get(key) != value:
                        meaningful_changes[key] = {"from": yesterdays_item.get(key), "to": value}
                if meaningful_changes:
                    changelog["modifications"].append({"bill_id": link.text, "changes": meaningful_changes})
    except Exception as e:
        print(f"❌ Error scraping {url}: {str(e)}")
        ## AN ERROR HAPPENED ON ONE OF THE PAGES
        ## CHECK IF YOU HAVE YESTERDAY'S DICT FROM THAT PAGE AND SAVE THAT INSTEAD
        if yesterdays_item:
            todays_bills.append(yesterdays_item)
        #UPDATE ERROR LOG
        error_log["errors"].append({
            "bill_id": link.text,
            "url": "https://www.palegis.us"+link['href'],
            "error_type": type(e).__name__,
            "message": str(e),
            "traceback": traceback.format_exc().splitlines()[-3:] # Keeps log clean by grabbing last few lines of trace
        })


#########STEP FIVE#########
#write files

with open(DATA_FILE, 'w') as f:
    # Sorting by 'id' so the order remains consistant from scrape to scrape
    sorted_data = sorted(todays_bills, key=lambda x: x['bill_id'])
    json.dump(sorted_data, f, indent=2)
if changelog["additions"] or changelog["deletions"] or changelog["modifications"]:
    with open(f"data/changelogs/{TODAY_STR}.json", 'w') as f:
        json.dump(changelog, f, indent=2)

# 3. Write Daily Error Log (Only write if errors occurred)
if error_log["errors"]:
    with open(f"data/error_logs/{TODAY_STR}.json", 'w') as f:
        json.dump(error_log, f, indent=2)
        
print("Scrape done!")